In [2]:
import pandas as pd
import numpy as np
import pickle

In [6]:
with open("../svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)

U = svd_model["U"]
sigma = svd_model["sigma"]
Vt = svd_model["Vt"]

print("SVD model loaded successfully!")

SVD model loaded successfully!


In [3]:
with open("../ctr_model.pkl", "rb") as f:
    ctr_model = pickle.load(f)

print("CTR model loaded successfully!")

CTR model loaded successfully!


In [4]:
movies = pd.read_csv("../ml-25m/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
print(svd_model.keys())

dict_keys(['U', 'sigma', 'Vt', 'user_map', 'movie_map'])


In [8]:
ratings = pd.read_csv("../ml-25m/ratings.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [9]:
print(movies.shape)
print(ratings.shape)

(62423, 3)
(25000095, 4)


In [10]:
ctr_features = pd.read_csv("../ctr_features.csv")

ctr_features.head()

,userId,movieId,rating,timestamp,avg_rating,rating_count,avg_movie_rating,movie_rating_count
0,99476,104374,3.5,1467897440,3.583333,6,4.048387,93
1,107979,2634,4.0,994007728,3.937500,16,3.111111,18
2,155372,1614,3.0,1097887531,2.863636,11,3.269608,102
3,65225,7153,4.0,1201382275,4.000000,2,4.142063,989
4,79161,500,5.0,1488915363,4.076923,13,3.451994,677


In [11]:
ctr_features.columns

Index(['userId', 'movieId', 'rating', 'timestamp', 'avg_rating',
       'rating_count', 'avg_movie_rating', 'movie_rating_count'],
      dtype='str')

In [12]:
feature_columns = [
    "avg_rating",
    "rating_count",
    "avg_movie_rating",
    "movie_rating_count"
]

X = ctr_features[feature_columns]

ctr_features["ctr_probability"] = ctr_model.predict_proba(X)[:, 1]

ctr_features.head()

,userId,movieId,rating,timestamp,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability
0,99476,104374,3.5,1467897440,3.583333,6,4.048387,93,0.732198
1,107979,2634,4.0,994007728,3.937500,16,3.111111,18,0.522959
2,155372,1614,3.0,1097887531,2.863636,11,3.269608,102,0.117718
3,65225,7153,4.0,1201382275,4.000000,2,4.142063,989,0.880884
4,79161,500,5.0,1488915363,4.076923,13,3.451994,677,0.720920


In [13]:
ctr_features["cf_score"] = ctr_features["rating"] / 5.0

ctr_features.head()

,userId,movieId,rating,timestamp,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability,cf_score
0,99476,104374,3.5,1467897440,3.583333,6,4.048387,93,0.732198,0.7
1,107979,2634,4.0,994007728,3.937500,16,3.111111,18,0.522959,0.8
2,155372,1614,3.0,1097887531,2.863636,11,3.269608,102,0.117718,0.6
3,65225,7153,4.0,1201382275,4.000000,2,4.142063,989,0.880884,0.8
4,79161,500,5.0,1488915363,4.076923,13,3.451994,677,0.720920,1.0


In [14]:
alpha = 0.7
beta = 0.3

ctr_features["hybrid_score"] = (
    alpha * ctr_features["cf_score"] +
    beta * ctr_features["ctr_probability"]
)

ctr_features.head()


,userId,movieId,rating,timestamp,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability,cf_score,hybrid_score
0,99476,104374,3.5,1467897440,3.583333,6,4.048387,93,0.732198,0.7,0.709659
1,107979,2634,4.0,994007728,3.937500,16,3.111111,18,0.522959,0.8,0.716888
2,155372,1614,3.0,1097887531,2.863636,11,3.269608,102,0.117718,0.6,0.455315
3,65225,7153,4.0,1201382275,4.000000,2,4.142063,989,0.880884,0.8,0.824265
4,79161,500,5.0,1488915363,4.076923,13,3.451994,677,0.720920,1.0,0.916276


In [15]:
hybrid_recommendations = ctr_features.sort_values(
    by="hybrid_score",
    ascending=False
)

hybrid_recommendations.head(20)

,userId,movieId,rating,timestamp,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability,cf_score,hybrid_score
1503,75309,128167,5.0,1558738163,5.0,116,5.0,1,0.997305,1.0,0.999191
383464,75309,176437,5.0,1558738306,5.0,116,5.0,1,0.997305,1.0,0.999191
465825,75309,92904,5.0,1558738198,5.0,116,5.0,1,0.997305,1.0,0.999191
53256,75309,164218,5.0,1558738309,5.0,116,5.0,1,0.997305,1.0,0.999191
473966,75309,101904,5.0,1558738193,5.0,116,5.0,1,0.997305,1.0,0.999191
301891,75309,83089,5.0,1558738168,5.0,116,5.0,1,0.997305,1.0,0.999191
314876,75309,201412,5.0,1558738318,5.0,116,5.0,1,0.997305,1.0,0.999191
476493,75309,91302,5.0,1558738190,5.0,116,5.0,1,0.997305,1.0,0.999191
96095,75309,174399,5.0,1558738206,5.0,116,5.0,1,0.997305,1.0,0.999191
110861,75309,124797,5.0,1558738303,5.0,116,5.0,1,0.997305,1.0,0.999191


In [16]:
final_recommendations = hybrid_recommendations.merge(
    movies,
    on="movieId",
    how="left"
)

final_recommendations[
    [
        "userId",
        "movieId",
        "title",
        "hybrid_score"
    ]
].head(20)

,userId,movieId,title,hybrid_score
0,75309,128167,Beer for My Horses (2008),0.999191
1,75309,176437,England Is Mine (2017),0.999191
2,75309,92904,We Were Here (2011),0.999191
3,75309,164218,Hurok (2016),0.999191
4,75309,101904,Happy (2011),0.999191
5,75309,83089,Violet Tendencies (2010),0.999191
6,75309,201412,Bitter Melon (2018),0.999191
7,75309,91302,"Tree, The (2010)",0.999191
8,75309,174399,Daddy's Little Girl (2012),0.999191
9,75309,124797,Voodoo Possession (2014),0.999191


In [17]:
final_recommendations.to_csv(
    "../hybrid_recommendations.csv",
    index=False
)

print("Hybrid recommendations saved successfully!")


Hybrid recommendations saved successfully!


In [18]:
print("U shape:", U.shape)
print("Sigma shape:", sigma.shape)
print("Vt shape:", Vt.shape)

U shape: (55057, 50)
Sigma shape: (50, 50)
Vt shape: (50, 10262)


In [20]:
user_map = svd_model["user_map"]
movie_map = svd_model["movie_map"]

In [21]:
print("Number of users:", len(user_map))
print("Number of movies:", len(movie_map))

Number of users: 55057
Number of movies: 10262


In [23]:
sample_user = list(user_map.keys())[0]

print("MovieLens User ID:", sample_user)

MovieLens User ID: 99476


In [24]:
user_index = user_map[sample_user]

print("User Index:", user_index)

User Index: 0


In [25]:
predicted_ratings = U[user_index] @ sigma @ Vt

print(predicted_ratings.shape)

(10262,)


In [26]:
cf_predictions = pd.DataFrame({
    "movie_index": range(len(predicted_ratings)),
    "cf_score": predicted_ratings
})

cf_predictions.head()

,movie_index,cf_score
0,0,0.001872
1,1,0.000004
2,2,0.000060
3,3,-0.000233
4,4,0.001516


In [27]:
reverse_movie_map = {
    v: k
    for k, v in movie_map.items()
}

cf_predictions["movieId"] = cf_predictions["movie_index"].map(
    reverse_movie_map
)

cf_predictions.head()

,movie_index,cf_score,movieId
0,0,0.001872,104374
1,1,0.000004,2634
2,2,0.000060,1614
3,3,-0.000233,7153
4,4,0.001516,500


In [28]:
cf_predictions = cf_predictions.merge(
    movies,
    on="movieId",
    how="left"
)

cf_predictions.head()


,movie_index,cf_score,movieId,title,genres
0,0,0.001872,104374,About Time (2013),Drama|Fantasy|Romance
1,1,0.000004,2634,"Mummy, The (1959)",Horror
2,2,0.000060,1614,In & Out (1997),Comedy
3,3,-0.000233,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
4,4,0.001516,500,Mrs. Doubtfire (1993),Comedy|Drama


In [29]:
cf_top100 = cf_predictions.sort_values(
    by="cf_score",
    ascending=False
).head(100)

cf_top100

,movie_index,cf_score,movieId,title,genres
599,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
756,756,0.042784,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
397,397,0.025901,3578,Gladiator (2000),Action|Adventure|Drama
423,423,0.023701,1193,One Flew Over the Cuckoo's Nest (1975),Drama
21,21,0.020826,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy
...,...,...,...,...,...
168,168,0.000821,805,"Time to Kill, A (1996)",Drama|Thriller
772,772,0.000816,5010,Black Hawk Down (2001),Action|Drama|War
1320,1320,0.000813,908,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller
1200,1200,0.000812,1278,Young Frankenstein (1974),Comedy|Fantasy


In [30]:
top100_features = cf_top100.merge(
    ctr_features[
        [
            "movieId",
            "avg_rating",
            "rating_count",
            "avg_movie_rating",
            "movie_rating_count"
        ]
    ],
    on="movieId",
    how="left"
)

top100_features.head()

,movie_index,cf_score,movieId,title,genres,avg_rating,rating_count,avg_movie_rating,movie_rating_count
0,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.250,4,3.83671,839
1,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,5.000,1,3.83671,839
2,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,3.600,5,3.83671,839
3,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.875,4,3.83671,839
4,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,4.000,2,3.83671,839


In [31]:
top100_features = top100_features.drop_duplicates(subset="movieId")

print(top100_features.shape)

top100_features.head()

(100, 9)


,movie_index,cf_score,movieId,title,genres,avg_rating,rating_count,avg_movie_rating,movie_rating_count
0,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.250000,4,3.836710,839
839,756,0.042784,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,5.000000,2,3.919007,1068
1907,397,0.025901,3578,Gladiator (2000),Action|Adventure|Drama,5.000000,2,3.992958,923
2830,423,0.023701,1193,One Flew Over the Cuckoo's Nest (1975),Drama,3.750000,2,4.228859,745
3575,21,0.020826,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,3.423077,13,4.064991,1054


In [32]:
X_ctr = top100_features[
    [
        "avg_rating",
        "rating_count",
        "avg_movie_rating",
        "movie_rating_count"
    ]
]

print(X_ctr.shape)

(100, 4)


In [33]:
top100_features["ctr_probability"] = ctr_model.predict_proba(X_ctr)[:, 1]

top100_features.head()

,movie_index,cf_score,movieId,title,genres,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability
0,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.250000,4,3.836710,839,0.081192
839,756,0.042784,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,5.000000,2,3.919007,1068,0.978248
1907,397,0.025901,3578,Gladiator (2000),Action|Adventure|Drama,5.000000,2,3.992958,923,0.981245
2830,423,0.023701,1193,One Flew Over the Cuckoo's Nest (1975),Drama,3.750000,2,4.228859,745,0.836360
3575,21,0.020826,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,3.423077,13,4.064991,1054,0.639422


In [34]:
alpha = 0.7
beta = 0.3

top100_features["hybrid_score"] = (
    alpha * top100_features["cf_score"] +
    beta * top100_features["ctr_probability"]
)

top100_features.head()

,movie_index,cf_score,movieId,title,genres,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability,hybrid_score
0,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.250000,4,3.836710,839,0.081192,0.066015
839,756,0.042784,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,5.000000,2,3.919007,1068,0.978248,0.323423
1907,397,0.025901,3578,Gladiator (2000),Action|Adventure|Drama,5.000000,2,3.992958,923,0.981245,0.312504
2830,423,0.023701,1193,One Flew Over the Cuckoo's Nest (1975),Drama,3.750000,2,4.228859,745,0.836360,0.267499
3575,21,0.020826,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,3.423077,13,4.064991,1054,0.639422,0.206405


In [35]:
final_top10 = top100_features.sort_values(
    by="hybrid_score",
    ascending=False
).head(10)

final_top10[
    [
        "movieId",
        "title",
        "cf_score",
        "ctr_probability",
        "hybrid_score"
    ]
]

,movieId,title,cf_score,ctr_probability,hybrid_score
839,1210,Star Wars: Episode VI - Return of the Jedi (1983),0.042784,0.978248,0.323423
1907,3578,Gladiator (2000),0.025901,0.981245,0.312504
29063,1258,"Shining, The (1980)",0.001542,0.981378,0.295493
33001,91529,"Dark Knight Rises, The (2012)",0.001378,0.975299,0.293554
40564,904,Rear Window (1954),0.001150,0.967154,0.290952
41954,1961,Rain Man (1988),0.001061,0.963163,0.289692
11106,293,Léon: The Professional (a.k.a. The Professiona...,0.008513,0.935948,0.286743
45164,1200,Aliens (1986),0.000997,0.948948,0.285382
45771,750,Dr. Strangelove or: How I Learned to Stop Worr...,0.000971,0.910016,0.273685
57540,44191,V for Vendetta (2006),0.000824,0.895857,0.269334


In [36]:
final_top10.to_csv("../final_hybrid_top10.csv", index=False)

print("Final Top 10 Hybrid Recommendations saved successfully!")

Final Top 10 Hybrid Recommendations saved successfully!


In [37]:
top100_features.to_csv("../top100_hybrid_candidates.csv", index=False)

print("Top 100 candidates saved successfully!")

Top 100 candidates saved successfully!


# Day 11 — Hybrid Scoring

## Objective
Combine Collaborative Filtering (SVD) scores and CTR probabilities to generate the final hybrid recommendations.

## Work Completed
- Loaded the trained SVD model.
- Generated predicted CF scores for a selected user.
- Selected the Top 100 candidate movies based on SVD scores.
- Retrieved CTR features for the candidate movies.
- Predicted click probabilities using the Logistic Regression model.
- Applied the hybrid scoring formula:
  
  Hybrid Score = α × CF Score + β × CTR Probability

- Ranked movies by hybrid score.
- Generated the final Top 10 recommendations.
- Saved the final recommendation list.

## Files Generated
- final_hybrid_top10.csv

## Outcome
A hybrid recommendation model was successfully implemented by combining Collaborative Filtering predictions with CTR probabilities to produce personalized movie recommendations.